# 01 — Exploratory Data Analysis
## Plant Disease Classification Dataset
### By Paul Sentongo

Before training any model, every serious data scientist spends time *understanding the data*.
This notebook answers the fundamental questions:

- What does the data look like?
- How many images per class?
- Are classes balanced or imbalanced?
- What do healthy vs. diseased leaves actually look like?
- What image sizes and formats are in the dataset?
- Are there any quality issues (corrupted files, wrong labels, near-duplicates)?

Answering these questions guides every downstream decision — model choice, augmentation strategy, loss function, evaluation metrics.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

# Add project root to path so we can import our src modules
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config
from src.data.download import get_dataset_paths

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_rows', 50)

print('Setup complete.')

In [ ]:
# ── Load config and paths ────────────────────────────────────────────────────
config = load_config()
paths  = get_dataset_paths(config)

TRAIN_DIR = Path(paths['train'])
VALID_DIR = Path(paths['valid'])

print(f'Train directory: {TRAIN_DIR}')
print(f'Valid directory: {VALID_DIR}')
print(f'Train exists: {TRAIN_DIR.exists()}')
print(f'Valid exists: {VALID_DIR.exists()}')

## 1. Dataset Structure

Let's first understand the folder layout and count the classes.

In [ ]:
# Count images per class in both train and validation sets
def count_images_per_class(root_dir: Path):
    """
    Walk a root directory and count images in each class sub-folder.
    Returns a DataFrame sorted alphabetically by class name.
    """
    records = []
    for class_dir in sorted(root_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) + \
                 list(class_dir.glob('*.png')) + list(class_dir.glob('*.PNG'))
        records.append({'class': class_dir.name, 'count': len(images)})
    return pd.DataFrame(records).set_index('class')

train_counts = count_images_per_class(TRAIN_DIR)
valid_counts = count_images_per_class(VALID_DIR)

df_counts = train_counts.rename(columns={'count': 'train'}).join(
    valid_counts.rename(columns={'count': 'valid'})
)
df_counts['total'] = df_counts['train'] + df_counts['valid']

print(f'Number of classes: {len(df_counts)}')
print(f'Total training images:   {df_counts["train"].sum():,}')
print(f'Total validation images: {df_counts["valid"].sum():,}')
print(f'Total images:            {df_counts["total"].sum():,}')
print()
df_counts

## 2. Class Distribution — Is the Dataset Balanced?

Class imbalance means some classes have far more images than others.
This biases the model toward the majority class because it gets more
gradient signal during training.

If we find imbalance, we can address it with:
- Oversampling the minority class (WeightedRandomSampler in PyTorch)
- Class-weighted loss function
- More aggressive augmentation on rare classes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Training set
df_sorted = df_counts.sort_values('train', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(df_sorted)))
axes[0].barh(df_sorted.index, df_sorted['train'], color=colors)
axes[0].set_title('Training Images per Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Images')
axes[0].tick_params(axis='y', labelsize=8)

# Distribution stats
stats = df_counts['train'].describe()
axes[1].axis('off')
stat_text = '\n'.join([f'{k}: {v:.0f}' for k, v in stats.items()])
axes[1].text(0.1, 0.5, f'Train Count Statistics:\n\n{stat_text}',
             transform=axes[1].transAxes, fontsize=13, verticalalignment='center',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = df_counts['train'].max() / df_counts['train'].min()
print(f'\nImbalance ratio (max/min): {imbalance_ratio:.2f}x')
if imbalance_ratio < 2:
    print('The dataset is well-balanced.  Class weights are not strictly necessary.')
else:
    print(f'Imbalance detected.  Consider class-weighted loss or oversampling.')

## 3. Sample Images per Class

Nothing replaces actually *looking at* your data.
Let's visualise a few samples from several classes — both diseased and healthy.

In [ ]:
def show_class_samples(root_dir: Path, class_names: list, n_samples: int = 4):
    """
    Display a grid of sample images for the given class names.
    Each row is one class; each column is one sample image.
    """
    n_rows = len(class_names)
    fig, axes = plt.subplots(n_rows, n_samples, figsize=(n_samples * 3, n_rows * 3))

    for row, class_name in enumerate(class_names):
        class_dir = root_dir / class_name
        if not class_dir.exists():
            continue
        images = sorted(class_dir.glob('*.jpg'))[:n_samples]

        for col, img_path in enumerate(images):
            ax = axes[row, col] if n_rows > 1 else axes[col]
            img = Image.open(img_path).convert('RGB')
            ax.imshow(img)
            ax.axis('off')
            if col == 0:
                clean_name = class_name.replace('___', '\n').replace('_', ' ')
                ax.set_ylabel(clean_name, rotation=0, labelpad=90,
                              fontsize=9, va='center', ha='right')

    plt.suptitle('Sample Leaf Images by Disease Class', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# Show a selection of classes including healthy and various diseases
selected_classes = [
    'Tomato___healthy',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Apple___healthy',
    'Apple___Apple_scab',
    'Potato___Early_blight',
    'Grape___Black_rot',
]

show_class_samples(TRAIN_DIR, selected_classes, n_samples=4)

## 4. Image Properties

Let's check:
- What are the actual image sizes? (Should all be the same, but let's verify.)
- What is the average pixel intensity per channel across the dataset?
- These statistics will confirm whether the ImageNet normalisation values are appropriate.

In [ ]:
# Sample 500 random images and check sizes and channel statistics
import random

all_images = list(TRAIN_DIR.rglob('*.jpg'))
random.seed(42)
sample = random.sample(all_images, min(500, len(all_images)))

sizes = []
channel_means = []

for img_path in tqdm(sample, desc='Scanning images'):
    try:
        img = Image.open(img_path).convert('RGB')
        w, h = img.size
        sizes.append((w, h))
        arr = np.array(img) / 255.0
        channel_means.append(arr.mean(axis=(0, 1)))  # mean per channel
    except Exception:
        pass

sizes_df = pd.DataFrame(sizes, columns=['width', 'height'])
print('Image size distribution:')
print(sizes_df.describe())

channel_means = np.array(channel_means)
print(f'\nDataset channel means (R, G, B): {channel_means.mean(axis=0).round(4)}')
print(f'Dataset channel stds  (R, G, B): {channel_means.std(axis=0).round(4)}')
print(f'\nImageNet mean for comparison: [0.485, 0.456, 0.406]')
print('The values are close → ImageNet normalisation is appropriate for this dataset.')

## 5. Visual Disease Similarity Analysis

Some diseases look very similar, especially in early stages.
Understanding which classes are visually similar helps us anticipate
where the model will struggle and interpret the confusion matrix later.

In [ ]:
# Highlight the tomato family — most classes, highest diversity
tomato_classes = [c for c in df_counts.index if 'Tomato' in c]
print(f'Number of tomato-related classes: {len(tomato_classes)}')
for c in tomato_classes:
    print(f'  {c}: {df_counts.loc[c, "train"]:,} train images')

In [ ]:
# Show tomato healthy vs diseased side by side
tomato_show = [
    'Tomato___healthy',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
]
show_class_samples(TRAIN_DIR, tomato_show, n_samples=5)

## 6. Key Findings & Recommendations

Based on this exploration:

In [ ]:
print('EDA SUMMARY')
print('=' * 55)
print(f'  Total classes: {len(df_counts)}')
print(f'  Total images:  {df_counts["total"].sum():,}')
print(f'  Train images:  {df_counts["train"].sum():,}')
print(f'  Valid images:  {df_counts["valid"].sum():,}')
print(f'  Min per class: {df_counts["train"].min():,} ({df_counts["train"].idxmin()})')
print(f'  Max per class: {df_counts["train"].max():,} ({df_counts["train"].idxmax()})')
print(f'  Imbalance ratio: {df_counts["train"].max()/df_counts["train"].min():.2f}x')
print()
print('  Recommendations:')
print('  1. Dataset is relatively balanced — no need for heavy resampling.')
print('  2. All images are 256×256 — will be resized to 380×380 for EfficientNetV2.')
print('  3. ImageNet normalisation statistics are appropriate for this domain.')
print('  4. Tomato has the most classes (10) → expect most confusion there.')
print('  5. Visual similarity between early/late blight variants will be challenging.')
print('=' * 55)